# Seminar 2. Custom PyTorch Operators

# Building Models in PyTorch Through Composition

PyTorch models are built using **composition**.  
Instead of defining one large monolithic network, we construct models by combining smaller, reusable modules.

Each module can contain other modules, which allows us to build hierarchical and well-structured architectures.

---

## Composition

Composition means:

- A model is built from smaller blocks.
- Each block can contain multiple layers.
- Blocks can be reused in larger architectures.
- Complex models are created by stacking simpler components.

This keeps code:

- Modular  
- Reusable  
- Readable  
- Easy to extend  


## Key Ideas

- Inherit from `nn.Module`
- Define layers inside `__init__`
- Define computation in `forward()`
- Create reusable blocks
- Build larger models by combining blocks

---

## Example: Model Built from Two Blocks

Below is a simple example where:

- We define a reusable blocks: `LinearReLUBlock` and `LinearTanhBlock`
- The final model is composed of two such blocks


In [2]:
import torch
import torch.nn as nn


class LinearReLUBlock(nn.Module):
    def __init__(self, in_features: int, out_features: int):
        super().__init__()

        self.linear = nn.Linear(in_features, out_features)
        self.activation = nn.ReLU()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.linear(x)
        x = self.activation(x)
        return x


class LinearTanhBlock(nn.Module):
    def __init__(self, in_features: int, out_features: int):
        super().__init__()

        self.linear = nn.Linear(in_features, out_features)
        self.activation = nn.Tanh()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.linear(x)
        x = self.activation(x)
        return x


class CombinedModel(nn.Module):
    def __init__(self):
        super().__init__()

        self.block1 = LinearReLUBlock(4, 8)
        self.block2 = LinearTanhBlock(8, 8)
        self.output = nn.Linear(8, 2)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.block1(x)
        x = self.block2(x)
        x = self.output(x)
        return x


model = CombinedModel()
print(model)

CombinedModel(
  (block1): LinearReLUBlock(
    (linear): Linear(in_features=4, out_features=8, bias=True)
    (activation): ReLU()
  )
  (block2): LinearTanhBlock(
    (linear): Linear(in_features=8, out_features=8, bias=True)
    (activation): Tanh()
  )
  (output): Linear(in_features=8, out_features=2, bias=True)
)


# What `nn.Module` Enables

When we inherit from `nn.Module`, we automatically gain powerful functionality that works **recursively across all submodules**.

## What `nn.Module` Gives Us



### Parameter Registration

All layers assigned as attributes (e.g. `self.linear = nn.Linear(...)`) are:

- Automatically registered
- Collected in `model.parameters()`
- Included in `model.state_dict()`

This works **recursively** for all sub-blocks.

In [3]:
print("Registered parameters:")
for name, param in model.named_parameters():
    print(name, param.shape)


Registered parameters:
block1.linear.weight torch.Size([8, 4])
block1.linear.bias torch.Size([8])
block2.linear.weight torch.Size([8, 8])
block2.linear.bias torch.Size([8])
output.weight torch.Size([2, 8])
output.bias torch.Size([2])


### Automatic Gradient Tracking

During the forward pass:

- PyTorch dynamically builds a computation graph
- Calling `loss.backward()` computes gradients
- Gradients are stored in each parameter’s `.grad`

No manual graph management is required.

In [4]:
x = torch.randn(5, 4)
target = torch.randn(5, 2)

criterion = nn.MSELoss()
output = model(x)
loss = criterion(output, target)

loss.backward()

print("\nGradient computed for output layer:",
      model.output.weight.grad is not None)
model.output.weight.grad


Gradient computed for output layer: True


tensor([[-0.0362, -0.0281,  0.1679, -0.0437, -0.1692, -0.0883, -0.1531, -0.1088],
        [ 0.0691,  0.1071, -0.0967,  0.0991,  0.0820,  0.0877,  0.1147,  0.0581]])

### Device and Type Transfer (`.to()`)

Calling:

    model.to(device)

or

    model.to(dtype)

moves all:

- Parameters
- Buffers
- Submodules

to CPU/GPU/dtype automatically.

In [5]:
import torch

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

dtype = torch.float32

model = model.to(device=device, dtype=dtype)

x: torch.Tensor = torch.randn(5, 4, device=device, dtype=dtype)

print("Model device:", next(model.parameters()).device)
print("Model dtype:", next(model.parameters()).dtype)

Model device: cpu
Model dtype: torch.float32


### Saving & Loading (`state_dict()`)

- `model.state_dict()` returns all parameters recursively
- `model.load_state_dict(...)` restores them

This works across the full module tree.

In [6]:
state_dict = model.state_dict()
torch.save(state_dict, "combined_model.pt")

# `train()` vs `eval()` Mode in PyTorch

PyTorch modules have two main modes: **training mode** and **evaluation mode**.  
Switching between them affects layers that behave differently during training and inference.

---

## `model.train()`

- Sets the model to **training mode**.
- Used when training the model with gradient updates.
- Affects certain layers, such as:

| Layer Type        | Behavior in `train()` Mode                  |
|------------------|--------------------------------------------|
| `Dropout`         | Randomly zeroes some activations           |
| `BatchNorm`       | Updates running statistics (mean/variance) |

- Gradients are computed as usual.

---

## `model.eval()`

- Sets the model to **evaluation (inference) mode**.
- Used when evaluating or deploying the model.
- Affects certain layers:

| Layer Type        | Behavior in `eval()` Mode                   |
|------------------|--------------------------------------------|
| `Dropout`         | Passes all activations through unchanged  |
| `BatchNorm`       | Uses stored running mean/variance         |

- No layers update internal statistics.
- Gradients are usually not required (often used with `torch.no_grad()`).

---

## Key Points

- Always use `model.train()` during training.
- Always use `model.eval()` during evaluation or testing.
- Forgetting to switch can lead to inconsistent results, especially with `Dropout` or `BatchNorm`.



In [7]:
import torch
import torch.nn as nn

# Simple model with Dropout and BatchNorm
class SimpleModel(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.fc1 = nn.Linear(4, 8)
        self.bn = nn.BatchNorm1d(8)
        self.dropout = nn.Dropout(p=0.5)
        self.fc2 = nn.Linear(8, 2)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.fc1(x)
        x = self.bn(x)
        x = torch.relu(x)
        x = self.dropout(x)
        x = self.fc2(x)
        return x


model = SimpleModel()
x = torch.randn(5, 4)

# Training mode
model.train()
output_train = model(x)
print("Training mode output:\n", output_train)

# Evaluation mode
model.eval()
with torch.no_grad():
    output_eval = model(x)
print("Evaluation mode output:\n", output_eval)


Training mode output:
 tensor([[ 1.5562, -1.0843],
        [ 1.4275,  0.6393],
        [ 0.0544, -0.2396],
        [ 0.2278, -0.6562],
        [ 0.2586, -0.1675]], grad_fn=<AddmmBackward0>)
Evaluation mode output:
 tensor([[ 0.2976, -0.2268],
        [ 0.3162,  0.0535],
        [ 0.2300, -0.0485],
        [ 0.2026, -0.1002],
        [ 0.2432, -0.0106]])


# `torch.no_grad()` and `torch.inference_mode()` in PyTorch

When performing inference (evaluating a model without updating parameters), PyTorch provides context managers to **disable gradient tracking**. This saves memory and speeds up computation.

---

## `torch.no_grad()`

- Disables gradient tracking.
- Useful during evaluation or inference.
- Gradients are **not computed**, but autograd still tracks operations for some internal purposes.
- Can be used as a **context manager** or a **function decorator**.

---

## `torch.inference_mode()`

- Introduced in PyTorch 1.9.
- Similar to `no_grad()`, but **more efficient**.
- Completely disables autograd and reduces memory usage.
- Recommended for pure inference pipelines.



In [8]:
import torch
import torch.nn as nn

class SimpleModel(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.fc = nn.Linear(4, 2)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.fc(x)

    # -----------------------------
    # Using torch.no_grad() as method decorator
    # -----------------------------
    @torch.no_grad()
    def forward_no_grad(self, x: torch.Tensor) -> torch.Tensor:
        return self.fc(x)

    # -----------------------------
    # Using torch.inference_mode() as method decorator
    # -----------------------------
    @torch.inference_mode()
    def forward_inference(self, x: torch.Tensor) -> torch.Tensor:
        return self.fc(x)


model = SimpleModel()
x = torch.randn(5, 4)

# -----------------------------
# Call decorated methods
# -----------------------------
output_no_grad_method = model.forward_no_grad(x)
output_infer_method = model.forward_inference(x)

print("Output no_grad method:\n", output_no_grad_method)
print("Output inference_mode method:\n", output_infer_method)

# -----------------------------
# Using context managers
# -----------------------------

with torch.no_grad():
    output_no_grad_cm = model(x)

with torch.inference_mode():
    output_infer_cm = model(x)

print("Output no_grad context manager:\n", output_no_grad_cm)
print("Output inference_mode context manager:\n", output_infer_cm)


Output no_grad method:
 tensor([[-0.1682, -1.0443],
        [-0.8392, -0.6108],
        [-1.2566, -0.4269],
        [ 0.7908, -0.6973],
        [-0.5566, -0.7536]])
Output inference_mode method:
 tensor([[-0.1682, -1.0443],
        [-0.8392, -0.6108],
        [-1.2566, -0.4269],
        [ 0.7908, -0.6973],
        [-0.5566, -0.7536]])
Output no_grad context manager:
 tensor([[-0.1682, -1.0443],
        [-0.8392, -0.6108],
        [-1.2566, -0.4269],
        [ 0.7908, -0.6973],
        [-0.5566, -0.7536]])
Output inference_mode context manager:
 tensor([[-0.1682, -1.0443],
        [-0.8392, -0.6108],
        [-1.2566, -0.4269],
        [ 0.7908, -0.6973],
        [-0.5566, -0.7536]])


# Disabling Gradients with `requires_grad_(False)`

PyTorch provides a convenient method `requires_grad_()` that can **enable or disable gradients in-place** for all parameters of a model or a tensor.

Using:

```python
param.requires_grad_(False)
```

- Sets `requires_grad=False` **in-place** for that parameter.
- This is useful for freezing models during inference or transfer learning.
- Can be applied to an entire model recursively by iterating over its parameters.


In [9]:
model = SimpleModel()

# Disable gradient computation for all parameters using requires_grad_()
for param in model.parameters():
    param.requires_grad_(False)

# Verify
for name, param in model.named_parameters():
    print(f"{name}: requires_grad={param.requires_grad}")

# Forward pass still works
x = torch.randn(5, 4)
output = model(x)
print("Output shape:", output.shape)

fc.weight: requires_grad=False
fc.bias: requires_grad=False
Output shape: torch.Size([5, 2])


# Redefining `train()` and `eval()` in `nn.Module`

PyTorch’s `nn.Module` provides built-in `train(mode: bool = True)` and `eval()` methods to switch between **training** and **evaluation** modes.  

Sometimes, when creating **custom modules or blocks**, you might want to **override these methods** to perform extra actions whenever the mode changes.

---

## Why Override?

- Apply mode-specific logic to sub-blocks or attributes that are not standard layers
- Log or track mode switches
- Automatically modify internal flags or buffers along with training/eval mode

---

## How It Works

- `train(mode: bool = True)` sets `self.training = mode` for the module
- `eval()` is equivalent to `train(False)`
- Default implementation recursively calls `train(mode)` on all submodules
- Overriding allows custom behavior while keeping recursive updates intact


In [10]:
import torch
import torch.nn as nn
from torch import Tensor
from typing import Self

class CustomBlock(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.linear = nn.Linear(4, 4)
        self.dropout = nn.Dropout(p=0.5)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.linear(x)
        x = torch.relu(x)
        x = self.dropout(x)
        return x

    # -----------------------------
    # Override train() method
    # -----------------------------
    def train(self, mode: bool = True) -> Self:
        print(f"CustomBlock set to {'train' if mode else 'eval'} mode")
        super().train(mode)  # Call original method to update submodules
        # Add any custom logic here
        return self

    # -----------------------------
    # Override eval() method
    # -----------------------------
    def eval(self) -> Self:
        print("CustomBlock set to eval mode")
        return super().eval()


# Example usage
model = CustomBlock()
x = torch.randn(2, 4)

# Switch to training mode
model.train()
output_train = model(x)

# Switch to evaluation mode
model.eval()
with torch.no_grad():
    output_eval = model(x)

print("Output training mode:", output_train)
print("Output eval mode:", output_eval)


CustomBlock set to train mode
CustomBlock set to eval mode
CustomBlock set to eval mode
Output training mode: tensor([[0.0000, 0.0000, 0.0000, 0.1534],
        [0.5624, 0.7759, 0.0000, 0.0000]], grad_fn=<MulBackward0>)
Output eval mode: tensor([[1.8985, 0.0000, 0.0000, 0.0767],
        [0.2812, 0.3879, 0.0000, 0.0000]])


# Common Module Aggregators in PyTorch

When building neural networks, it is often useful to group multiple layers or submodules together.  
PyTorch provides several **module aggregators** that help organize layers and blocks. The most common ones are:



## `nn.Sequential`

- Holds modules in a sequential order.
- Executes them **in the order they are added** during the forward pass.
- Ideal for simple **stacked layers** with a single input and output.

**Key points:**

- Forward pass is automatically defined.
- Cannot handle multiple inputs or branching.

In [11]:
seq_model = nn.Sequential(
    nn.Linear(4, 8),
    nn.ReLU(),
    nn.Linear(8, 2)
)

x = torch.randn(5, 4)
output_seq = seq_model(x)
print("nn.Sequential output shape:", output_seq.shape)


nn.Sequential output shape: torch.Size([5, 2])


## `nn.ModuleList`

- Holds a **list of modules**.
- Does **not define a forward pass automatically**.
- Useful when you need to **loop over modules**, or have conditional computation.

**Key points:**

- Modules are registered properly, so parameters are tracked.
- You must define your own `forward()`.

In [12]:
class ModuleListModel(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.layers = nn.ModuleList([
            nn.Linear(4, 8),
            nn.ReLU(),
            nn.Linear(8, 2)
        ])

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        for layer in self.layers:
            x = layer(x)
        return x

ml_model = ModuleListModel()
output_ml = ml_model(x)
print("nn.ModuleList output shape:", output_ml.shape)

nn.ModuleList output shape: torch.Size([5, 2])


## `nn.ModuleDict`

- Holds modules in a **dictionary** with string keys.
- Useful for architectures with **named branches**, **dynamic selection**, or **multi-head outputs**.
- Like `ModuleList`, it does **not define a forward pass**.

In [13]:
class ModuleDictModel(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.branches = nn.ModuleDict({
            "branch1": nn.Linear(4, 8),
            "branch2": nn.Linear(4, 8)
        })
        self.output: nn.Linear = nn.Linear(8, 2)

    def forward(self, x: torch.Tensor, branch_name: str = "branch1") -> torch.Tensor:
        x = self.branches[branch_name](x)
        return self.output(x)

md_model = ModuleDictModel()
output_md = md_model(x, branch_name="branch2")
print("nn.ModuleDict output shape:", output_md.shape)

nn.ModuleDict output shape: torch.Size([5, 2])


## Type hints and Shape Information

For production code, a practical pattern is to use `Tensor` for type hints and document the expected shape next to the signature.

`Tensor` tells us what kind of object is expected, while `[B, C, H, W]` documents the tensor dimensions explicitly.


In [14]:
from torch import Tensor

def normalize_images(x: Tensor) -> Tensor:
    # x: [B, C, H, W]
    return x / 255.0



## Работа на семинаре

Блоки из этой части дополнительно реализованы после семинара: хотелось самостоятельно потренироваться

### LSTM

![lstm](https://github.com/v3code/MTAI2026-Deep-learning-frameworks-seminars-autumn/blob/main/seminar_2/assets/LSTM.png?raw=1)

In [15]:
from typing import Optional
import torch
from torch import nn, Tensor


class LSTMCell(nn.Module):
    def __init__(self, input_size: int, hidden_size: int):
        super().__init__()
        self.hidden_size = hidden_size
        self.gates = nn.Linear(input_size + hidden_size, 4 * hidden_size)

    def forward(self, x: Tensor, state: Optional[tuple[Tensor, Tensor]] = None) -> tuple[Tensor, Tensor]:
        if state is None:
            h = x.new_zeros(x.size(0), self.hidden_size)
            c = x.new_zeros(x.size(0), self.hidden_size)
        else:
            h, c = state
        i, f, c_tilde, o = self.gates(torch.cat([x, h], dim=1)).chunk(4, dim=1)
        i = torch.sigmoid(i)
        f = torch.sigmoid(f)
        c_tilde = torch.tanh(c_tilde)
        o = torch.sigmoid(o)
        c = f * c + i * c_tilde
        h = o * torch.tanh(c)
        return h, c


class LSTM(nn.Module):
    def __init__(self, input_size: int, hidden_size: int):
        super().__init__()
        self.cell = LSTMCell(input_size, hidden_size)

    def forward(self, x: Tensor, state: Optional[tuple[Tensor, Tensor]] = None) -> tuple[Tensor, tuple[Tensor, Tensor]]:
        outputs = []
        for t in range(x.size(1)):
            state = self.cell(x[:, t], state)
            outputs.append(state[0])
        return torch.stack(outputs, dim=1), state

### Inception

![inception](https://github.com/v3code/MTAI2026-Deep-learning-frameworks-seminars-autumn/blob/main/seminar_2/assets/inception.png?raw=1)

In [16]:
class BasicConv2d(nn.Module):
    def __init__(self, in_channels: int, out_channels: int, kernel_size: int, padding: int = 0):
        super().__init__()
        self.conv = nn.Conv2d(in_channels, out_channels, kernel_size, padding=padding, bias=False)
        self.bn = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU()

    def forward(self, x: Tensor) -> Tensor:
        return self.relu(self.bn(self.conv(x)))


class InceptionBlock(nn.Module):
    def __init__(self, in_channels: int, ch1x1: int, ch3x3_reduce: int, ch3x3: int, ch5x5_reduce: int, ch5x5: int, pool_proj: int):
        super().__init__()
        self.branch1 = BasicConv2d(in_channels, ch1x1, kernel_size=1)
        self.branch2 = nn.Sequential(
            BasicConv2d(in_channels, ch3x3_reduce, kernel_size=1),
            BasicConv2d(ch3x3_reduce, ch3x3, kernel_size=3, padding=1),
        )
        self.branch3 = nn.Sequential(
            BasicConv2d(in_channels, ch5x5_reduce, kernel_size=1),
            BasicConv2d(ch5x5_reduce, ch5x5, kernel_size=5, padding=2),
        )
        self.branch4 = nn.Sequential(
            nn.MaxPool2d(kernel_size=3, stride=1, padding=1),
            BasicConv2d(in_channels, pool_proj, kernel_size=1),
        )

    def forward(self, x: Tensor) -> Tensor:
        return torch.cat([self.branch1(x), self.branch2(x), self.branch3(x), self.branch4(x)], dim=1)

### SE

![se](https://github.com/v3code/MTAI2026-Deep-learning-frameworks-seminars-autumn/blob/main/seminar_2/assets/SqueezeAndExcite.png?raw=1)

In [17]:
class SEBlock(nn.Module):
    def __init__(self, channels: int, reduction: int = 16):
        super().__init__()
        hidden = max(channels // reduction, 1)
        self.squeeze = nn.AdaptiveAvgPool2d(1)
        self.excitation = nn.Sequential(
            nn.Linear(channels, hidden),
            nn.ReLU(),
            nn.Linear(hidden, channels),
            nn.Sigmoid(),
        )

    def forward(self, x: Tensor) -> Tensor:
        B, C, _, _ = x.shape
        s = self.squeeze(x).view(B, C)
        w = self.excitation(s).view(B, C, 1, 1)
        return x * w

### Selective Kernel

![selective](https://github.com/v3code/MTAI2026-Deep-learning-frameworks-seminars-autumn/blob/main/seminar_2/assets/SelectiveKernel.png?raw=1)


In [18]:
class SKConv(nn.Module):
    def __init__(self, channels: int, kernel_sizes: tuple[int, ...] = (3, 5), reduction: int = 16, min_dim: int = 32, stride: int = 1, groups: int = 1):
        super().__init__()
        d = max(channels // reduction, min_dim)
        self.branches = nn.ModuleList([
            nn.Sequential(
                nn.Conv2d(channels, channels, k, stride=stride, padding=k // 2, groups=groups, bias=False),
                nn.BatchNorm2d(channels),
                nn.ReLU(),
            )
            for k in kernel_sizes
        ])
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(
            nn.Linear(channels, d, bias=False),
            nn.BatchNorm1d(d),
            nn.ReLU(),
        )
        self.selects = nn.ModuleList([nn.Linear(d, channels) for _ in kernel_sizes])

    def forward(self, x: Tensor) -> Tensor:
        feats = torch.stack([branch(x) for branch in self.branches], dim=1)
        u = feats.sum(dim=1)
        s = self.pool(u).flatten(1)
        z = self.fc(s)
        logits = torch.stack([select(z) for select in self.selects], dim=1)
        weights = torch.softmax(logits, dim=1)
        return (feats * weights[..., None, None]).sum(dim=1)

### PatchMerger

![patchmerger](https://github.com/v3code/MTAI2026-Deep-learning-frameworks-seminars-autumn/blob/main/seminar_2/assets/PatchMerger.png?raw=1)


In [19]:
class PatchMerger(nn.Module):
    def __init__(self, dim: int, num_tokens_out: int):
        super().__init__()
        self.norm = nn.LayerNorm(dim)
        self.queries = nn.Parameter(torch.randn(num_tokens_out, dim) * dim ** -0.5)

    def forward(self, x: Tensor) -> Tensor:
        x = self.norm(x)
        attn = torch.softmax(self.queries @ x.transpose(1, 2), dim=-1)
        return attn @ x

In [20]:
# проверочная ячейка

torch.manual_seed(0)

x = torch.randn(4, 7, 16)
lstm = LSTM(16, 32)
ref = nn.LSTM(16, 32, batch_first=True)
with torch.no_grad():
    lstm.cell.gates.weight.copy_(torch.cat([ref.weight_ih_l0, ref.weight_hh_l0], dim=1))
    lstm.cell.gates.bias.copy_(ref.bias_ih_l0 + ref.bias_hh_l0)
out, (h, c) = lstm(x)
ref_out, (ref_h, ref_c) = ref(x)
print(out.shape, torch.allclose(out, ref_out, atol=1e-5), torch.allclose(c, ref_c[0], atol=1e-5))

print(InceptionBlock(192, 64, 96, 128, 16, 32, 32)(torch.randn(2, 192, 28, 28)).shape)

feat = torch.randn(2, 64, 16, 16)
print(nn.Sequential(nn.Conv2d(3, 64, 3, padding=1), SEBlock(64))(torch.randn(2, 3, 16, 16)).shape)
print(SKConv(64)(feat).shape, SKConv(64, stride=2)(feat).shape)

print(PatchMerger(64, 8)(torch.randn(2, 196, 64)).shape)

torch.Size([4, 7, 32]) True True
torch.Size([2, 256, 28, 28])
torch.Size([2, 64, 16, 16])
torch.Size([2, 64, 16, 16]) torch.Size([2, 64, 8, 8])
torch.Size([2, 8, 64])


## Домашнее задание

## Задание 1, реализуйте требуемые блоки

## ResNet Block (0.1 балл)

![Resnet](https://github.com/v3code/MTAI2026-Deep-learning-frameworks-seminars-autumn/blob/main/seminar_2/assets/ResBlock.png?raw=1)

https://arxiv.org/pdf/1512.03385

In [21]:
class ResidualBlock(nn.Module):
    def __init__(self, in_channels: int, out_channels: int, stride: int = 1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU()
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels),
            )
        else:
            self.shortcut = nn.Identity()

    def forward(self, x: Tensor) -> Tensor:
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        return self.relu(out + self.shortcut(x))

## Depthwise Separable Convolution (0.1 балл)
![DepthWiseConv](https://github.com/v3code/MTAI2026-Deep-learning-frameworks-seminars-autumn/blob/main/seminar_2/assets/DepthWiseConv.png?raw=1)

https://arxiv.org/pdf/1610.02357

In [22]:
class SeparableConv2d(nn.Module):
    def __init__(self, in_channels: int, out_channels: int, kernel_size: int = 3, stride: int = 1, padding: int = 1, dilation: int = 1, bias: bool = True):
        super().__init__()
        self.depthwise = nn.Conv2d(in_channels, in_channels, kernel_size, stride=stride, padding=padding, dilation=dilation, groups=in_channels, bias=bias)
        self.pointwise = nn.Conv2d(in_channels, out_channels, kernel_size=1, bias=bias)

    def forward(self, x: Tensor) -> Tensor:
        return self.pointwise(self.depthwise(x))

## Vanilla Attention (0.1 балл)

Let:

$$
\text{query} \in \mathbb{R}^{B \times d} \\
\text{key} \in \mathbb{R}^{B \times L \times d}
$$

---

### Alignment Scores

$$
\text{score} = \text{key} \cdot (W_\text{align} \, \text{query})^T \\
\text{score} \in \mathbb{R}^{B \times L}
$$

---

### Attention Weights

$$
\text{att} = \text{softmax}(\text{score}, \text{dim}=1) \\
\text{att} \in \mathbb{R}^{B \times L}
$$

---

### Context Vector

$$
\text{context} = \sum_{i=1}^{L} \text{att}_i \cdot \text{key}_i \\
\text{context} \in \mathbb{R}^{B \times d}
$$

---

### Output

$$
\text{out} = \tanh(W_\text{value} \, \text{context} + W_\text{query} \, \text{query}) \\
\text{out} \in \mathbb{R}^{B \times d}
$$



https://arxiv.org/abs/1409.0473


https://arxiv.org/abs/1508.04025

In [23]:
from typing import Optional
import torch
from torch import nn, Tensor


class VanillaAttention(nn.Module):
    def __init__(self, dim: int):
        super().__init__()
        self.W_align = nn.Linear(dim, dim, bias=False)
        self.W_value = nn.Linear(dim, dim, bias=False)
        self.W_query = nn.Linear(dim, dim, bias=False)

    def forward(self, query: Tensor, key: Tensor, mask: Optional[Tensor] = None) -> tuple[Tensor, Tensor]:
        score = torch.bmm(key, self.W_align(query).unsqueeze(-1)).squeeze(-1)
        if mask is not None:
            score = score.masked_fill(~mask.bool(), float("-inf"))
        att = torch.softmax(score, dim=1)
        context = torch.bmm(att.unsqueeze(1), key).squeeze(1)
        out = torch.tanh(self.W_value(context) + self.W_query(query))
        return out, att

## Dot Product Attention (0.1 балл)

$$
Q \in \mathbb{R}^{B \times L_q \times d_k} \\
K \in \mathbb{R}^{B \times L_k \times d_k} \\
V \in \mathbb{R}^{B \times L_k \times d_k}
$$

$$
S = \frac{Q K^T}{\sqrt{d_k}}
$$

$$
\text{Attention}(Q, K, V) = \text{softmax}(S, \text{dim}=-1) \, V
$$



https://arxiv.org/abs/1706.03762


In [24]:
import math


class ScaledDotProductAttention(nn.Module):
    def __init__(self, dropout: float = 0.0):
        super().__init__()
        self.dropout = nn.Dropout(dropout)

    def forward(self, q: Tensor, k: Tensor, v: Tensor, mask: Optional[Tensor] = None) -> tuple[Tensor, Tensor]:
        scores = q @ k.transpose(-2, -1) / math.sqrt(q.size(-1))
        if mask is not None:
            scores = scores.masked_fill(~mask.bool(), float("-inf"))
        attn = self.dropout(torch.softmax(scores, dim=-1))
        return attn @ v, attn

## Multihead Attention (0.1 балл)

![MultiheadAttention](https://github.com/v3code/MTAI2026-Deep-learning-frameworks-seminars-autumn/blob/main/seminar_2/assets/MultiheadAttention.webp?raw=1)

https://arxiv.org/abs/1706.03762


In [25]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model: int, num_heads: int, dropout: float = 0.0):
        super().__init__()
        assert d_model % num_heads == 0
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)
        self.attention = ScaledDotProductAttention(dropout)

    def _split_heads(self, x: Tensor) -> Tensor:
        B, L, _ = x.shape
        return x.view(B, L, self.num_heads, self.d_k).transpose(1, 2)

    def forward(self, query: Tensor, key: Tensor, value: Tensor, mask: Optional[Tensor] = None) -> tuple[Tensor, Tensor]:
        B, L_q, _ = query.shape
        q = self._split_heads(self.W_q(query))
        k = self._split_heads(self.W_k(key))
        v = self._split_heads(self.W_v(value))
        if mask is not None:
            if mask.dim() == 2:
                mask = mask[:, None, None, :]
            elif mask.dim() == 3:
                mask = mask[:, None, :, :]
        out, attn = self.attention(q, k, v, mask)
        out = out.transpose(1, 2).reshape(B, L_q, self.num_heads * self.d_k)
        return self.W_o(out), attn

## Transformer Encoder Layer (0.1 балл)


![Transformer Encoder Layer](https://github.com/v3code/MTAI2026-Deep-learning-frameworks-seminars-autumn/blob/main/seminar_2/assets/TransformerEncoder.png?raw=1)


https://arxiv.org/abs/1706.03762

In [26]:
class TransformerEncoderLayer(nn.Module):
    def __init__(self, d_model: int, num_heads: int, d_ff: int = 2048, dropout: float = 0.1):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, num_heads, dropout)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model),
        )
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)

    def forward(self, x: Tensor, mask: Optional[Tensor] = None) -> Tensor:
        attn_out, _ = self.self_attn(x, x, x, mask)
        x = self.norm1(x + self.dropout1(attn_out))
        x = self.norm2(x + self.dropout2(self.ffn(x)))
        return x

## MLP Mixer (0.1 балл)


![MLPMixer](https://github.com/v3code/MTAI2026-Deep-learning-frameworks-seminars-autumn/blob/main/seminar_2/assets/MLPMixer.png?raw=1)


https://arxiv.org/abs/2105.01601

In [27]:
class MlpBlock(nn.Module):
    def __init__(self, dim: int, hidden_dim: int, dropout: float = 0.0):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, dim),
            nn.Dropout(dropout),
        )

    def forward(self, x: Tensor) -> Tensor:
        return self.net(x)


class MLPMixerBlock(nn.Module):
    def __init__(self, num_patches: int, dim: int, tokens_hidden_dim: int, channels_hidden_dim: int, dropout: float = 0.0):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.token_mixing = MlpBlock(num_patches, tokens_hidden_dim, dropout)
        self.norm2 = nn.LayerNorm(dim)
        self.channel_mixing = MlpBlock(dim, channels_hidden_dim, dropout)

    def forward(self, x: Tensor) -> Tensor:
        y = self.norm1(x).transpose(1, 2)
        x = x + self.token_mixing(y).transpose(1, 2)
        x = x + self.channel_mixing(self.norm2(x))
        return x

## ConvMixer (0.1 балл)

![ConvMixer](https://github.com/v3code/MTAI2026-Deep-learning-frameworks-seminars-autumn/blob/main/seminar_2/assets/ConvMixer.png?raw=1)


https://arxiv.org/abs/2201.09792

In [28]:
class ConvMixerLayer(nn.Module):
    def __init__(self, dim: int, kernel_size: int = 9):
        super().__init__()
        self.depthwise = nn.Sequential(
            nn.Conv2d(dim, dim, kernel_size, groups=dim, padding="same"),
            nn.GELU(),
            nn.BatchNorm2d(dim),
        )
        self.pointwise = nn.Sequential(
            nn.Conv2d(dim, dim, kernel_size=1),
            nn.GELU(),
            nn.BatchNorm2d(dim),
        )

    def forward(self, x: Tensor) -> Tensor:
        x = x + self.depthwise(x)
        return self.pointwise(x)


class ConvMixer(nn.Module):
    def __init__(self, dim: int, depth: int, kernel_size: int = 9, patch_size: int = 7, in_channels: int = 3, num_classes: int = 1000):
        super().__init__()
        self.patch_embed = nn.Sequential(
            nn.Conv2d(in_channels, dim, kernel_size=patch_size, stride=patch_size),
            nn.GELU(),
            nn.BatchNorm2d(dim),
        )
        self.layers = nn.Sequential(*[ConvMixerLayer(dim, kernel_size) for _ in range(depth)])
        self.head = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(dim, num_classes),
        )

    def forward(self, x: Tensor) -> Tensor:
        return self.head(self.layers(self.patch_embed(x)))

In [29]:
# проверочная ячейка

import torch.nn.functional as F

torch.manual_seed(0)

x = torch.randn(2, 16, 32, 32)
print(ResidualBlock(16, 16)(x).shape, ResidualBlock(16, 32, stride=2)(x).shape)
print(SeparableConv2d(16, 32)(x).shape)

q = torch.randn(2, 64)
k = torch.randn(2, 10, 64)
mask = torch.ones(2, 10, dtype=torch.bool)
mask[1, 7:] = False
out, att = VanillaAttention(64)(q, k, mask)
print(out.shape, att.shape, att[1, 7:])

Q, K, V = torch.randn(2, 5, 64), torch.randn(2, 10, 64), torch.randn(2, 10, 64)
out, _ = ScaledDotProductAttention()(Q, K, V)
print(torch.allclose(out, F.scaled_dot_product_attention(Q, K, V), atol=1e-5))

mha = MultiHeadAttention(64, 8)
ref = nn.MultiheadAttention(64, 8, batch_first=True)
with torch.no_grad():
    ref.in_proj_weight.copy_(torch.cat([mha.W_q.weight, mha.W_k.weight, mha.W_v.weight]))
    ref.in_proj_bias.copy_(torch.cat([mha.W_q.bias, mha.W_k.bias, mha.W_v.bias]))
    ref.out_proj.weight.copy_(mha.W_o.weight)
    ref.out_proj.bias.copy_(mha.W_o.bias)
print(torch.allclose(mha(Q, K, V, mask)[0], ref(Q, K, V, key_padding_mask=~mask)[0], atol=1e-5))

print(TransformerEncoderLayer(64, 8, d_ff=128)(torch.randn(2, 10, 64), mask).shape)
print(MLPMixerBlock(49, 64, 32, 256)(torch.randn(2, 49, 64)).shape)
print(ConvMixer(dim=64, depth=2, num_classes=10)(torch.randn(2, 3, 224, 224)).shape)

torch.Size([2, 16, 32, 32]) torch.Size([2, 32, 16, 16])
torch.Size([2, 32, 32, 32])
torch.Size([2, 64]) torch.Size([2, 10]) tensor([0., 0., 0.], grad_fn=<SliceBackward0>)
True
True
torch.Size([2, 10, 64])
torch.Size([2, 49, 64])
torch.Size([2, 10])


## Вопрос (0.2 балла)

Объясните, почему MLPMixer, ConvMixer может работать почти так же эффективно, как обычный Multihead Attention.

Напишите формулу, связывающую Multihead Attention, ConvMixer и MLPMixer

Опишите преимущества и недостатки между ConvMixer, MLPMixer и Multihead Attention

---

Ответ:

**Почему Mixer'ы работают почти как attention.** Все три блока устроены одинаково: сначала информация смешивается между токенами (патчами), потом между каналами, и всё это обёрнуто в residual-связи и нормализацию. Отличаются они только способом смешивания токенов. В attention веса считаются заново для каждого входа, в MLP-Mixer это обучаемая, но фиксированная матрица на все патчи, а в ConvMixer это фиксированное локальное ядро. Получается, что для картинок главное - сама схема «патчи + чередование token mixing и channel mixing», а адаптивность весов в attention даёт меньше выигрыша, чем можно было ожидать. На большом датасете фиксированная матрица выучивает «усреднённые» паттерны внимания, а ConvMixer набирает глобальный контекст за счёт глубины и большого ядра

**Общая формула**

Пусть $X \in \mathbb{R}^{N \times C}$ - это $N$ токенов по $C$ каналов. Смешивание токенов во всех трёх случаях записывается как

$$Y = A \, X W_V, \qquad Y_i = \sum_{j=1}^{N} A_{ij} \, (X W_V)_j$$

Отличаются три блока только матрицей $A$:

$$\text{MHA:} \quad A^{(h)} = \mathrm{softmax}\left(\frac{X W_Q^{(h)} \left(X W_K^{(h)}\right)^T}{\sqrt{d_k}}\right)$$

$$\text{MLP-Mixer:} \quad A = W_2 W_1 \in \mathbb{R}^{N \times N}, \quad W_V = I$$

$$\text{ConvMixer:} \quad A^{(c)}_{ij} = \begin{cases} w^{(c)}_{i-j}, & |i - j| \le k/2 \\ 0, & \text{иначе} \end{cases}, \quad W_V = I$$

В MHA матрица $A$ зависит от входа и своя у каждой головы, а результаты голов потом конкатенируются и умножаются на $W_O$. В MLP-Mixer (если для простоты убрать GELU внутри token-MLP) $A$ - одна обучаемая плотная матрица на все каналы, и от входа она не зависит. В ConvMixer $A$ - ленточная матрица из весов depthwise-свёртки, где $i - j$ - это сдвиг между патчами на сетке. Она тоже не зависит от входа, смотрит только на соседей и своя для каждого канала. То есть ConvMixer похож на attention с $C$ головами по одному каналу и фиксированными локальными весами

**Плюсы и минусы**
- **Multihead Attention.** Плюсы: веса подстраиваются под конкретный вход, глобальный контекст есть уже на первом слое, длина входа может быть любой. Минусы: сложность $O(N^2)$ по памяти и времени, мало встроенных предположений о данных (слабый inductive bias), поэтому нужно много данных, и обязательно нужны позиционные эмбеддинги.
- **MLP-Mixer.** Плюсы: очень простой, внутри только матричные умножения, которые хорошо ложатся на GPU, смешивание сразу глобальное, сложность линейная по $N$. Минусы: число патчей зашито в веса, поэтому разрешение входа менять нельзя; веса не зависят от входа; без большого датасета или сильной регуляризации он быстро переобучается.
- **ConvMixer.** Плюсы: за счёт локальности и инвариантности к сдвигу хорошо учится даже на небольших данных, работает с любым разрешением, стоит $O(N k^2 C)$. Минусы: рецептивное поле локальное, и глобальный контекст появляется только с глубиной; веса тоже не адаптивные; модель завязана на сетку, поэтому в основном подходит для картинок